# Wayzyy's Dynamic Pricing Engine - V1 (London)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

## 1. Load Data & Filter for London

In [ ]:
df = pd.read_csv('../data/raw/rental-price-predict/airbnb_train.csv')
df_london = df[df['city'] == 'London'].copy()
print(f'London dataset size: {len(df_london)}')

## 2 & 3. EDA and Data Cleaning
Removing outliers ($0 or > $1000) and missing target values.

In [ ]:
if df_london['price'].dtype == 'O':
    df_london['price'] = df_london['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

plt.figure(figsize=(10,6))
sns.histplot(df_london['price'], bins=50, kde=True)
plt.title('Price Distribution Before Cleaning')
plt.show()

# Clean outliers
df_london = df_london[(df_london['price'] > 0) & (df_london['price'] <= 1000)]
df_london = df_london.dropna(subset=['price'])

plt.figure(figsize=(10,6))
sns.histplot(df_london['price'], bins=50, kde=True)
plt.title('Price Distribution After Cleaning')
plt.show()

## 4 & 5. Feature Selection and Preprocessing

In [ ]:
features = ['property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'number_of_reviews', 'review_scores_rating']
target = 'price'

X = df_london[features]
y = df_london[target]

numeric_features = ['accommodates', 'bathrooms', 'bedrooms', 'beds', 'number_of_reviews', 'review_scores_rating']
categorical_features = ['property_type', 'room_type']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 7 & 8. Model Training and Evaluation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGB_AVAILABLE:
    models['XGBoost'] = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)

results = {}
best_model = None
best_r2 = -float('inf')
best_name = ''

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    
    if r2 > best_r2:
        best_r2 = r2
        best_name = name
        best_model = pipeline

results_df = pd.DataFrame(results).T
print(results_df)

## 9. Actual vs Predicted

In [ ]:
y_pred_best = best_model.predict(X_test)
comparison_df = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred_best})
print(comparison_df.head(10))

## 10. Save Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_pricing_model_v1.pkl')
print('Model saved to ../models/best_pricing_model_v1.pkl')